In [1]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.3 MB/s eta 0:00:00


In [2]:
import tensorflow as tf
import gradio as gr
import numpy as np
from PIL import Image
import requests
from urllib.request import urlretrieve

# Download human-readable labels for ImageNet.
response = requests.get("https://git.io/JJkYN")
labels = response.text.split("\n")

mobile_net = tf.keras.applications.MobileNetV2()
inception_net = tf.keras.applications.InceptionV3()


def classify_image(im, model_type):
    """Classifies an image using either MobileNet or InceptionNet.

    Args:
        im: The input image.
        model_type: The type of model to use ("mobilenet" or "inceptionnet").

    Returns:
        A dictionary of predicted labels and their probabilities.
    """
    if model_type == "mobilenet":
        im = Image.fromarray(im.astype("uint8"), "RGB")
        im = im.resize((224, 224))
        arr = np.array(im).reshape((-1, 224, 224, 3))
        arr = tf.keras.applications.mobilenet.preprocess_input(arr)
        model = mobile_net
    elif model_type == "inceptionnet":
        im = Image.fromarray(im.astype("uint8"), "RGB")
        im = im.resize((299, 299))
        arr = np.array(im).reshape((-1, 299, 299, 3))
        arr = tf.keras.applications.inception_v3.preprocess_input(arr)
        model = inception_net
    else:
        raise ValueError("Invalid model_type. Choose 'mobilenet' or 'inceptionnet'.")

    prediction = model.predict(arr).flatten()
    return {labels[i]: float(prediction[i]) for i in range(1000)}


imagein = gr.Image()
model_type = gr.Radio(["mobilenet", "inceptionnet"], label="Model Type")  # Added radio button for model selection
label = gr.Label(num_top_classes=3)
sample_images = [
    ["monkey.jpg"],
    ["sailboat.jpg"],
    ["bicycle.jpg"],
    ["fox.jpg"],
]

iface = gr.Interface(
    fn=lambda im, model_type: classify_image(im, model_type),  # Pass both image and model type to the function
    inputs=[imagein, model_type],  # Include model_type as an input
    outputs=label,
    title="MobileNet vs. InceptionNet",
    description="""Compare 2 state-of-the-art machine learning models:
    a lightweight model (MobileNet) that has an accuracy of 0.704, vs.
    InceptionNet, a much heavier model that has an accuracy of 0.779.""",
    examples=sample_images,
)

iface.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab85d61e7899201e39.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import tensorflow as tf
import gradio as gr
import numpy as np
from PIL import Image
import requests
import logging

# Set up logging for debugging and error tracking
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Download ImageNet labels for human-readable class names
try:
    response = requests.get("https://git.io/JJkYN")
    labels = response.text.split("\n")
    logging.info("Successfully downloaded ImageNet labels")
except Exception as e:
    logging.error(f"Failed to download ImageNet labels: {e}")
    raise

# Load pre-trained models: MobileNetV2 and InceptionV3
try:
    mobile_net = tf.keras.applications.MobileNetV2(weights='imagenet')
    inception_net = tf.keras.applications.InceptionV3(weights='imagenet')
    logging.info("Successfully loaded MobileNetV2 and InceptionV3 models")
except Exception as e:
    logging.error(f"Failed to load models: {e}")
    raise

def classify_image_with_mobile_net(im):
    """
    Classify an input image using MobileNetV2.

    Args:
        im: Input image as a numpy array (RGB format).

    Returns:
        Dictionary mapping ImageNet class labels to confidence scores.
    """
    try:
        # Validate input image
        if im is None or not isinstance(im, np.ndarray):
            logging.error("Invalid input: Image is None or not a numpy array")
            return {"Error": "Invalid image input"}

        # Convert to PIL Image and preprocess for MobileNetV2
        im = Image.fromarray(im.astype('uint8'), 'RGB')
        im = im.resize((224, 224))  # MobileNetV2 input size
        arr = np.array(im).reshape((-1, 224, 224, 3))
        arr = tf.keras.applications.mobilenet_v2.preprocess_input(arr)

        # Make prediction
        prediction = mobile_net.predict(arr, verbose=0).flatten()
        logging.info("MobileNetV2 prediction completed")

        # Return top predictions as a dictionary
        return {labels[i]: float(prediction[i]) for i in range(len(labels))}
    except Exception as e:
        logging.error(f"MobileNetV2 classification failed: {e}")
        return {"Error": f"Classification failed: {str(e)}"}

def classify_image_with_inception_net(im):
    """
    Classify an input image using InceptionV3.

    Args:
        im: Input image as a numpy array (RGB format).

    Returns:
        Dictionary mapping ImageNet class labels to confidence scores.
    """
    try:
        # Validate input image
        if im is None or not isinstance(im, np.ndarray):
            logging.error("Invalid input: Image is None or not a numpy array")
            return {"Error": "Invalid image input"}

        # Convert to PIL Image and preprocess for InceptionV3
        im = Image.fromarray(im.astype('uint8'), 'RGB')
        im = im.resize((299, 299))  # InceptionV3 input size
        arr = np.array(im).reshape((-1, 299, 299, 3))
        arr = tf.keras.applications.inception_v3.preprocess_input(arr)

        # Make prediction
        prediction = inception_net.predict(arr, verbose=0).flatten()
        logging.info("InceptionV3 prediction completed")

        # Return top predictions as a dictionary
        return {labels[i]: float(prediction[i]) for i in range(len(labels))}
    except Exception as e:
        logging.error(f"InceptionV3 classification failed: {e}")
        return {"Error": f"Classification failed: {str(e)}"}

def compare_models(im):
    """
    Compare predictions from MobileNetV2 and InceptionV3 for a given image.

    Args:
        im: Input image as a numpy array (RGB format).

    Returns:
        Tuple of dictionaries: (MobileNetV2 predictions, InceptionV3 predictions).
    """
    mobile_net_results = classify_image_with_mobile_net(im)
    inception_net_results = classify_image_with_inception_net(im)
    return mobile_net_results, inception_net_results

# Define Gradio interface components
image_input = gr.Image(label="Upload an Image")
label_output = gr.Label(num_top_classes=3, label="Top Predictions")

# Create two separate Label instances for each model's output
mobile_net_label = gr.Label(num_top_classes=3, label="Top Predictions (MobileNetV2)")
inception_net_label = gr.Label(num_top_classes=3, label="Top Predictions (InceptionV3)")

# Sample images (using placeholder URLs; replace with actual URLs)
sample_images = [
    "monkey.jpg",  # Replace with actual image URLs
    "sailboat.jpg",
    "bicycle.jpg",
    "fox.jpg"
]

# Create and launch Gradio interface
try:
    gr.Interface(
        fn=compare_models,  # Single function that calls both models
        inputs=image_input,
        # Use the separate Label instances for each output
        outputs=[mobile_net_label, inception_net_label],
        title="MobileNetV2 vs. InceptionV3 Image Classification",
        description=(
            "Compare two state-of-the-art image classification models: "
            "MobileNetV2 (lightweight, top-1 accuracy: 70.4%) and InceptionV3 "
            "(larger, top-1 accuracy: 77.9%). Upload an image to see top-3 predictions "
            "from both models side by side."
        ),
        examples=sample_images,
        flagging_mode="never"  # Updated from deprecated allow_flagging
    ).launch(debug=True)
    logging.info("Gradio interface launched successfully")
except Exception as e:
    logging.error(f"Failed to launch Gradio interface: {e}")
    raise

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://62f5bffb7a730ac801.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 2146, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 1664, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/anyio/to_thread.py", line 56, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
